# Diagnóstico da rodada 2 (exploratório): vale a pena usar o endereço do cadastro municipal?

**Objetivo:** medir, só com contagens, quanto a rodada 2 (localizar de novo os CNPJs de confiança baixa ou muito baixa pelo endereço do cadastro municipal) poderia ganhar. **Não altera nenhum arquivo do fluxo** e não grava nada: só logs na tela e no arquivo de log.

**Entradas:** `df_cnpj_espacial` (08) e `df_cadastro` (03).

**Perguntas (A a D):**
- **A.** Quantos CNPJs têm confiança baixa ou muito baixa (e de qual passo vieram)?
- **B.** Desses, quantos têm inscrição no cadastro municipal (cruzando pelo CNPJ)?
- **C.** Desses, em quantos o endereço do cadastro é **diferente** do da Receita? (Se for igual, rodar a cascata de novo daria o mesmo resultado.)
- **D.** Dos que têm endereço diferente, quantos têm rua, número e CEP em formato utilizável e cidade São José dos Campos?

**Como ler o resultado:** o número de **D** é o teto de quantos CNPJs poderiam mudar de ponto. Quanto desse teto vira ganho de confiança só se descobre simulando a cascata, o que este notebook **não** faz.

**Fora do fluxo:** não entra no `executar_tudo.py`. A saída tem só contagens (sem coordenadas, CPF ou nomes).

**Seções:** 1. Grupo de confiança baixa (A) · 2. Cruzamento com o cadastro (B) · 3. Comparação dos endereços (C e D) · 4. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Biblioteca de tabelas (DataFrames).
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns do projeto e as de endereço da cascata (as mesmas regras de normalização do 05).
from src import config, fontes, registro
from src.enderecos import aplicar_nucleo, limpar_via, normalizar_numero, normalizar_texto, tirar_numero_do_fim
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("exploratorio_diagnostico_rodada2_cadastro", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# ===== PARÂMETROS =====
# Confianças que a rodada 2 tentaria melhorar.
CONFIANCAS_ALVO = ["baixa", "muito baixa"]
# Nome dos passos mais fracos da cascata (para separar o grupo A por origem).
NOME_PASSO = {13: "passo 13 (OpenStreetMap, só a rua)", 14: "passo 14 (centro do bairro)", 15: "passo 15 (centro do município)"}
# Como a cidade de São José dos Campos fica depois de normalizada (maiúsculas, sem acento e sem barra).
CIDADE_MUNICIPIO = "SAO JOSE DOS CAMPOS SP"

# ===== ENTRADAS =====
df_cnpj_espacial = fontes.ler_processamento(cfg, "df_cnpj_espacial")   # 08
df_cadastro = fontes.ler_processamento(cfg, "df_cadastro")             # 03

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. Grupo de confiança baixa (A)
Separa os CNPJs com confiança baixa ou muito baixa e mostra de qual passo da cascata eles vêm. Os passos 13, 14 e 15 (OpenStreetMap, centro do bairro e centro do município) são os mais fracos e os que mais ganhariam com um endereço melhor. Só logs.

In [ ]:
log.info("SEÇÃO 1: grupo de confiança baixa (A)...")
# Guarda a hora de início para calcular o tempo total da seção.
inicio_s1 = time.time()

# Verdadeiro para quem tem a confiança baixa ou muito baixa.
grupo_alvo = df_cnpj_espacial["confianca"].isin(CONFIANCAS_ALVO)
# Quantidade total do grupo A.
total_alvo = int(grupo_alvo.sum())
log.info(f"    CNPJs com confiança {' ou '.join(CONFIANCAS_ALVO)}: {total_alvo:,} de {len(df_cnpj_espacial):,}")

# Confere se a coluna 'passo' existe (ela diz de qual passo da cascata veio cada ponto).
if "passo" not in df_cnpj_espacial.columns:
    raise KeyError("Seção interrompida: a coluna 'passo' não existe no df_cnpj_espacial. Nada foi alterado.")

# Nome do grupo de origem de cada CNPJ: os passos fracos têm nome próprio e o resto vai em 'outros passos'.
origem = df_cnpj_espacial["passo"].map(NOME_PASSO).fillna("outros passos (via, CEP, interpolação...)")
# Conta o grupo A por origem e mostra.
log.info("    Por origem do ponto:")
for nome, qtd in origem[grupo_alvo].value_counts().items():
    log.info(f"        {nome}: {qtd:,}")
# Mostra também por confiança, para conferir com o log do 07.
for conf, qtd in df_cnpj_espacial.loc[grupo_alvo, "confianca"].value_counts().items():
    log.info(f"    Confiança {conf}: {qtd:,}")

log.info(f"SEÇÃO 1 concluída! {formatar_tempo(time.time() - inicio_s1)}")

## 2. Cruzamento com o cadastro (B)
Cruza o grupo A com o `df_cadastro` pelo CNPJ de 14 dígitos e conta quantos têm inscrição municipal, por origem do ponto e por situação cadastral. Só logs.

In [ ]:
log.info("SEÇÃO 2: cruzamento com o cadastro municipal (B)...")
# Guarda a hora de início para calcular o tempo total da seção.
inicio_s2 = time.time()

# Conjunto dos CNPJs que existem no cadastro (busca rápida).
cnpjs_cadastro = set(df_cadastro["cnpj"].astype(str))
# Verdadeiro para quem tem inscrição no cadastro municipal.
tem_cadastro = df_cnpj_espacial["cnpj"].astype(str).isin(cnpjs_cadastro)
# Grupo B: confiança baixa E com cadastro.
grupo_b = grupo_alvo & tem_cadastro
log.info(f"    Do grupo A, com cadastro municipal: {int(grupo_b.sum()):,} de {total_alvo:,} "
         f"({(grupo_b.sum() / max(total_alvo, 1)):.1%})")

# Por origem do ponto: quantos têm cadastro em cada um.
log.info("    Por origem do ponto (com cadastro / total):")
for nome in origem[grupo_alvo].value_counts().index:
    sel = grupo_alvo & (origem == nome)
    log.info(f"        {nome}: {int((sel & tem_cadastro).sum()):,} de {int(sel.sum()):,}")

# Por situação cadastral da Receita (baixados quase nunca estão no cadastro).
if "situacao_nome" in df_cnpj_espacial.columns:
    log.info("    Por situação cadastral (com cadastro / total):")
    for nome in df_cnpj_espacial.loc[grupo_alvo, "situacao_nome"].value_counts().index:
        sel = grupo_alvo & (df_cnpj_espacial["situacao_nome"] == nome)
        log.info(f"        {nome}: {int((sel & tem_cadastro).sum()):,} de {int(sel.sum()):,}")

log.info(f"SEÇÃO 2 concluída! {formatar_tempo(time.time() - inicio_s2)}")

## 3. Comparação dos endereços (C e D)
Para o grupo B, monta o endereço da Receita e o do cadastro **com as mesmas regras de normalização da cascata** (via sem acento, sem tipo duplicado etc.; número = primeiro grupo de dígitos) e compara. A via é comparada pelo **núcleo do nome** (sem tipo, título e preposição), para que o tipo escrito de outro jeito conte como a mesma via. Se o cadastro abreviar o tipo (ex.: 'AV'), a cascata não reconhece a abreviação e a contagem de C sai **inflada**; o log mostra as primeiras palavras mais comuns da rua do cadastro para conferir isso. No cadastro, a coluna `rua` vem como `NOME DA RUA, NÚMERO`: a via é o que vem antes da última vírgula e o número é o que vem depois.

- **C (endereço diferente):** a via **ou** o número diferem entre Receita e cadastro. Se só o CEP difere, também é contado à parte.
- **D (utilizável):** entre os diferentes, o cadastro tem via, número e CEP de 8 dígitos e a cidade é São José dos Campos/SP.

Só logs, nenhuma linha de dado é mostrada.

In [ ]:
log.info("SEÇÃO 3: comparação dos endereços (C e D)...")
# Guarda a hora de início para calcular o tempo total da seção.
inicio_s3 = time.time()

# ===================================================================
# ETAPA 1: ENDEREÇO DA RECEITA (grupo B)
# ===================================================================
log.info("Etapa 1: normalizando o endereço da Receita (grupo B)...")
# Só as linhas do grupo B, com a numeração própria.
rec = df_cnpj_espacial.loc[grupo_b].copy()
# CNPJ como texto, para cruzar com o cadastro.
rec["cnpj"] = rec["cnpj"].astype(str)
# Número normalizado (primeiro grupo de dígitos).
rec["num_rec"] = normalizar_numero(rec["numero"])
# Via normalizada: tipo + nome da via, sem acento e em maiúsculas.
via_rec = normalizar_texto(rec["tipo_logradouro"].fillna("") + " " + rec["logradouro"].fillna(""))
# Tira do fim da via o número do próprio imóvel, quando ele aparece lá (mesma regra do 05).
via_rec = tirar_numero_do_fim(via_rec, rec["num_rec"])
# Limpeza completa da via (mesmas regras das outras bases).
rec["via_rec"] = limpar_via(via_rec, "Receita").mask(lambda s: s == "")
# CEP sem espaços (vazio vira NaN).
cep_rec = rec["cep"].fillna("").astype(str).str.strip()
rec["cep_rec"] = cep_rec.mask(cep_rec == "")
log.info("Etapa 1 concluída!")

# ===================================================================
# ETAPA 2: ENDEREÇO DO CADASTRO
# ===================================================================
log.info("\nEtapa 2: normalizando o endereço do cadastro...")
# Só os CNPJs do grupo B.
cad = df_cadastro[df_cadastro["cnpj"].astype(str).isin(set(rec["cnpj"]))].copy()
cad["cnpj"] = cad["cnpj"].astype(str)
# A coluna 'rua' vem como 'NOME DA RUA, NÚMERO': separa na ÚLTIMA vírgula.
partes = cad["rua"].fillna("").astype(str).str.rsplit(",", n=1, expand=True)
# Se não houver vírgula, a segunda parte não existe: cria vazia.
if partes.shape[1] < 2:
    partes[1] = ""
# Via do cadastro: parte antes da vírgula, normalizada e limpa.
via_cad = normalizar_texto(partes[0])
cad["via_cad"] = limpar_via(via_cad, "Cadastro").mask(lambda s: s == "")
# Número do cadastro: parte depois da vírgula, normalizada.
cad["num_cad"] = normalizar_numero(partes[1])
# CEP do cadastro: só dígitos; vale se tiver 8.
cep_cad = cad["cep"].fillna("").astype(str).str.replace(r"\D", "", regex=True)
cad["cep_cad"] = cep_cad.mask(cep_cad.str.len() != 8)
# Cidade do cadastro normalizada (sem acento e sem a barra) para comparar com o município.
cad["cidade_ok"] = normalizar_texto(cad["cidade"]) == CIDADE_MUNICIPIO
# Como o tipo da via (RUA, AVENIDA, AV...) é escrito no cadastro: a primeira palavra mais comum da rua (todas as linhas).
# Serve para ver se o cadastro abrevia o tipo ('AV', 'R'); a cascata só reconhece o tipo escrito por extenso.
primeira_palavra = normalizar_texto(df_cadastro["rua"].fillna("").astype(str).str.rsplit(",", n=1).str[0]).str.split().str[0]
log.info("    Primeira palavra da rua no cadastro (as 12 mais comuns):")
for palavra, qtd in primeira_palavra.value_counts().head(12).items():
    log.info(f"        {palavra}: {qtd:,}")
log.info("Etapa 2 concluída!")

# ===================================================================
# ETAPA 3: COMPARAR
# ===================================================================
log.info("\nEtapa 3: comparando Receita x cadastro...")
# Junta as duas tabelas pelo CNPJ (uma linha por CNPJ nas duas).
comp = rec[["cnpj", "via_rec", "num_rec", "cep_rec"]].merge(
    cad[["cnpj", "via_cad", "num_cad", "cep_cad", "cidade_ok"]], on="cnpj", how="left")
# Confere que o cruzamento não criou linhas a mais.
if len(comp) != len(rec):
    raise ValueError(f"Seção interrompida: o cruzamento criou {len(comp) - len(rec):,} linhas a mais "
                     "(CNPJ repetido no cadastro). Nada foi alterado.")

def igual(a, b):
    """Verdadeiro quando os dois valores existem e são iguais (vazio nunca é igual a nada)."""
    return a.notna() & b.notna() & (a == b)

# Via e número iguais nas duas bases.
# A via é comparada pelo NÚCLEO do nome (sem o tipo, títulos e preposições): 'AV SAO JOAO' e 'AVENIDA SAO JOAO'
# contam como a mesma via. Assim a diferença de escrita do tipo não conta como endereço diferente.
via_igual = igual(aplicar_nucleo(comp["via_rec"]), aplicar_nucleo(comp["via_cad"]))
num_igual = igual(comp["num_rec"], comp["num_cad"])
cep_igual = igual(comp["cep_rec"], comp["cep_cad"])
# C: endereço diferente = a via OU o número diferem (inclui o caso de faltar um dos dois).
diferente = ~(via_igual & num_igual)
# Só o CEP difere (via e número iguais).
so_cep = (via_igual & num_igual) & ~cep_igual
# Totais do grupo B.
total_b = len(comp)
log.info(f"    Grupo B: {total_b:,} CNPJs")
log.info(f"    Via e número iguais, CEP igual (nada a ganhar): {int((~diferente & cep_igual).sum()):,}")
log.info(f"    Via e número iguais, CEP diferente: {int(so_cep.sum()):,}")
log.info(f"    C: via ou número diferentes: {int(diferente.sum()):,}")
# Detalha o C: o que difere.
log.info(f"        só o número difere (via igual): {int((via_igual & ~num_igual & diferente).sum()):,}")
log.info(f"        só a via difere (número igual): {int((~via_igual & num_igual & diferente).sum()):,}")
log.info(f"        via e número diferem: {int((~via_igual & ~num_igual).sum()):,}")

# D: entre os diferentes, o cadastro tem via, número, CEP de 8 dígitos e a cidade do município.
utilizavel = diferente & comp["via_cad"].notna() & comp["num_cad"].notna() & comp["cep_cad"].notna() & comp["cidade_ok"].fillna(False)
log.info(f"    D: diferentes e com via, número, CEP e cidade utilizáveis no cadastro: {int(utilizavel.sum()):,}")
# Por que os diferentes não são utilizáveis.
nao_util = diferente & ~utilizavel
log.info(f"        não utilizáveis: {int(nao_util.sum()):,}")
log.info(f"            sem via no cadastro: {int((nao_util & comp['via_cad'].isna()).sum()):,}")
log.info(f"            sem número no cadastro: {int((nao_util & comp['num_cad'].isna()).sum()):,}")
log.info(f"            sem CEP de 8 dígitos no cadastro: {int((nao_util & comp['cep_cad'].isna()).sum()):,}")
log.info(f"            cidade de fora do município: {int((nao_util & ~comp['cidade_ok'].fillna(False)).sum()):,}")
log.info("Etapa 3 concluída!")

log.info(f"SEÇÃO 3 concluída! {formatar_tempo(time.time() - inicio_s3)}")

## 4. Resumo
Funil de A a D, em uma linha cada. Só logs.

In [ ]:
log.info("RESUMO do diagnóstico da rodada 2...")
log.info(f"    A. Confiança baixa ou muito baixa: {total_alvo:,}")
log.info(f"    B. Com cadastro municipal: {int(grupo_b.sum()):,}")
log.info(f"    C. Com endereço diferente (via ou número): {int(diferente.sum()):,}")
log.info(f"    D. Diferente e utilizável (teto de CNPJs que poderiam mudar de ponto): {int(utilizavel.sum()):,} "
         f"({int(utilizavel.sum()) / max(total_alvo, 1):.1%} do grupo A)")
log.info("    Observação: D é só o teto. Quanto vira ganho de confiança só se descobre simulando a cascata com o endereço do cadastro.")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")